# Нейросетевой эксперимент по контролю качества данных

Этот ноутбук проверяет, как результаты автоэнкодера связаны с дальнейшим статистическим анализом.

В предыдущем ноутбуке автоэнкодер выделил строки с высокой ошибкой восстановления. Такие строки нельзя автоматически считать ошибочными, но их можно рассматривать как кандидатов на ручную проверку.

В этом ноутбуке выполняются два диагностических шага:

1. сравнение распределения целевой переменной у обычных и подозрительных строк;
2. сравнение качества моделей на полном датасете и на датасете без строк, помеченных автоэнкодером.

Цель ноутбука — показать, что нейросеть используется не только для прогноза, но и как инструмент статистической обработки и контроля качества информации.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.analysis.baseline_modeling import (
    choose_target,
    build_feature_frame,
    build_models,
    evaluate_models,
)
from src.parsers.common import PROCESSED_DIR, REPORTS_DIR

TABLES_DIR = REPORTS_DIR / "tables"
FIGURES_DIR = REPORTS_DIR / "figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

dataset_path = PROCESSED_DIR / "final_dataset_for_modeling.csv"
anomaly_scores_path = TABLES_DIR / "autoencoder_anomaly_scores.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Основной датасет:", dataset_path)
print("Результаты автоэнкодера:", anomaly_scores_path)


## 1. Загрузка данных

Загружаются две таблицы:

- основной подготовленный датасет `final_dataset_for_modeling.csv`;
- результаты автоэнкодера `autoencoder_anomaly_scores.csv`.

Результаты автоэнкодера используются только для диагностики. Они не подаются в модель как обычные признаки.


In [ ]:
df = pd.read_csv(dataset_path)
anomaly_scores = pd.read_csv(anomaly_scores_path)

overview = pd.DataFrame(
    [
        {
            "Таблица": "Основной датасет",
            "Строк": len(df),
            "Столбцов": df.shape[1],
        },
        {
            "Таблица": "Результаты автоэнкодера",
            "Строк": len(anomaly_scores),
            "Столбцов": anomaly_scores.shape[1],
        },
    ]
)

overview


## 2. Объединение основного датасета с результатами автоэнкодера

К основному датасету добавляются ошибка восстановления и признак подозрительной строки.

Важно: эти поля нужны только для разбиения строк на две группы. Они не должны использоваться как входные признаки модели, потому что иначе возникнет утечка диагностической информации.


In [ ]:
needed_columns = [
    "interval_id",
    "reconstruction_error",
    "anomaly_threshold",
    "is_autoencoder_anomaly",
]

missing = [column for column in needed_columns if column not in anomaly_scores.columns]
if missing:
    raise ValueError(f"В таблице автоэнкодера нет нужных колонок: {missing}")

if "interval_id" not in df.columns:
    raise ValueError("В основном датасете нет колонки interval_id. Нельзя объединить данные с результатами автоэнкодера.")

df_nn = df.merge(
    anomaly_scores[needed_columns],
    on="interval_id",
    how="left",
)

df_nn["is_autoencoder_anomaly"] = df_nn["is_autoencoder_anomaly"].fillna(False).astype(bool)

summary = pd.DataFrame(
    [
        {
            "Показатель": "Всего строк",
            "Значение": len(df_nn),
        },
        {
            "Показатель": "Строк с ошибкой восстановления",
            "Значение": int(df_nn["reconstruction_error"].notna().sum()),
        },
        {
            "Показатель": "Подозрительных строк автоэнкодера",
            "Значение": int(df_nn["is_autoencoder_anomaly"].sum()),
        },
        {
            "Показатель": "Доля подозрительных строк, %",
            "Значение": round(df_nn["is_autoencoder_anomaly"].mean() * 100, 2),
        },
    ]
)

summary


## 3. Распределение целевой переменной у обычных и подозрительных строк

Автоэнкодер не использует целевую переменную как готовый ответ. Поэтому полезно отдельно проверить, отличаются ли строки, найденные автоэнкодером, по интенсивности изменения береговой бровки.

Если подозрительные строки имеют более широкий разброс или более высокие значения целевой переменной, это может означать, что автоэнкодер выделяет статистически важную часть данных.

Это не доказывает ошибку в данных. Высокая ошибка восстановления может быть связана как с ошибкой наблюдения, так и с реальной особенностью участка или периода.


In [ ]:
target = choose_target(df_nn)

regular_target = pd.to_numeric(
    df_nn.loc[~df_nn["is_autoencoder_anomaly"], target],
    errors="coerce",
).dropna()

anomaly_target = pd.to_numeric(
    df_nn.loc[df_nn["is_autoencoder_anomaly"], target],
    errors="coerce",
).dropna()

target_summary = pd.DataFrame(
    [
        {
            "Группа": "Обычные строки",
            "Количество": len(regular_target),
            "Среднее": regular_target.mean(),
            "Медиана": regular_target.median(),
            "Стандартное отклонение": regular_target.std(),
            "Минимум": regular_target.min(),
            "Максимум": regular_target.max(),
        },
        {
            "Группа": "Подозрительные строки автоэнкодера",
            "Количество": len(anomaly_target),
            "Среднее": anomaly_target.mean(),
            "Медиана": anomaly_target.median(),
            "Стандартное отклонение": anomaly_target.std(),
            "Минимум": anomaly_target.min(),
            "Максимум": anomaly_target.max(),
        },
    ]
)

target_summary


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.boxplot(
    [regular_target, anomaly_target],
    labels=["Обычные строки", "Подозрительные строки"],
    showmeans=True,
)

ax.set_title("Целевая переменная у обычных и подозрительных строк")
ax.set_ylabel("Интенсивность изменения бровки, м/год")
ax.grid(axis="y", alpha=0.3)

output_path = FIGURES_DIR / "05_target_distribution_by_autoencoder_flag.png"
fig.savefig(output_path, dpi=240, bbox_inches="tight", facecolor="white")
plt.show()

print("Сохранено:", output_path)


## 4. Проверка влияния подозрительных строк на качество моделей

Теперь сравнивается качество моделей на двух вариантах датасета:

1. полный датасет;
2. датасет без строк, которые автоэнкодер пометил как подозрительные.

Это не означает, что подозрительные строки нужно удалить из итоговых данных. Эксперимент нужен только для оценки того, насколько такие строки влияют на моделирование.


In [ ]:
def evaluate_dataset_variant(data: pd.DataFrame, variant_name: str) -> pd.DataFrame:
    data = data.copy()

    service_columns = [
        "reconstruction_error",
        "anomaly_threshold",
        "is_autoencoder_anomaly",
    ]
    data = data.drop(columns=service_columns, errors="ignore")

    target = choose_target(data)
    X, y, metadata, numeric_features, categorical_features, excluded_columns = build_feature_frame(data, target)
    models = build_models(numeric_features, categorical_features)

    selected_model_names = [
        "DummyRegressor_median",
        "HistGradientBoostingRegressor",
        "MLPRegressor",
    ]

    selected_models = {
        name: model
        for name, model in models.items()
        if name in selected_model_names
    }

    missing_models = [name for name in selected_model_names if name not in selected_models]
    if missing_models:
        raise ValueError(f"В build_models нет ожидаемых моделей: {missing_models}")

    result = evaluate_models(
        X=X,
        y=y,
        metadata=metadata,
        models=selected_models,
        target=target,
    )

    metrics = result[0] if isinstance(result, tuple) else result

    metrics = metrics.copy()
    metrics.insert(0, "dataset_variant", variant_name)
    return metrics


metrics_full = evaluate_dataset_variant(
    df_nn,
    "Полный датасет",
)

metrics_without_anomalies = evaluate_dataset_variant(
    df_nn.loc[~df_nn["is_autoencoder_anomaly"]].copy(),
    "Без подозрительных строк автоэнкодера",
)

nn_quality_metrics = pd.concat(
    [metrics_full, metrics_without_anomalies],
    ignore_index=True,
)

output_metrics_path = TABLES_DIR / "05_neural_quality_filtering_metrics.csv"
nn_quality_metrics.to_csv(output_metrics_path, index=False)

print("Сохранено:", output_metrics_path)

display_columns = [
    "dataset_variant",
    "model",
    "n_rows",
    "test_mae",
    "test_rmse",
    "test_r2",
]

available_display_columns = [column for column in display_columns if column in nn_quality_metrics.columns]
nn_quality_metrics[available_display_columns]


In [ ]:
plot_df = nn_quality_metrics.copy()

model_order = [
    "DummyRegressor_median",
    "HistGradientBoostingRegressor",
    "MLPRegressor",
]

variant_order = [
    "Полный датасет",
    "Без подозрительных строк автоэнкодера",
]

x = np.arange(len(model_order))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))

for i, variant in enumerate(variant_order):
    values = (
        plot_df.loc[plot_df["dataset_variant"].eq(variant)]
        .set_index("model")
        .reindex(model_order)["test_mae"]
    )

    ax.bar(
        x + (i - 0.5) * width,
        values,
        width=width,
        label=variant,
    )

ax.set_title("Влияние строк, найденных автоэнкодером, на MAE моделей")
ax.set_ylabel("MAE, м/год")
ax.set_xticks(x)
ax.set_xticklabels(model_order, rotation=20, ha="right")
ax.grid(axis="y", alpha=0.3)
ax.legend()

output_path = FIGURES_DIR / "05_autoencoder_filtering_model_mae.png"
fig.savefig(output_path, dpi=240, bbox_inches="tight", facecolor="white")
plt.show()

print("Сохранено:", output_path)


## 5. Топ подозрительных строк

Ниже выводятся строки с наибольшей ошибкой восстановления.

Эта таблица нужна не для автоматического удаления данных, а для ручной проверки: участок, профиль, интервал наблюдения и возможные QC-пояснения нужно смотреть отдельно.


In [ ]:
top_anomalies_path = TABLES_DIR / "autoencoder_top_anomalies.csv"

if top_anomalies_path.exists():
    top_anomalies = pd.read_csv(top_anomalies_path)
    top_columns = [
        "rank",
        "interval_id",
        "site_id",
        "profile_id",
        "date_start",
        "date_end",
        "reconstruction_error",
        "anomaly_threshold",
        "is_autoencoder_anomaly",
        "anomaly_note_ru",
    ]
    available_top_columns = [column for column in top_columns if column in top_anomalies.columns]
    display(top_anomalies[available_top_columns].head(10))
else:
    print(f"Файл не найден: {top_anomalies_path}")


## 6. Вывод

В этом ноутбуке автоэнкодер использовался как нейросетевой инструмент диагностики данных.

Результаты показали, что строки, помеченные автоэнкодером как нетипичные, нельзя автоматически считать ошибочными. После их исключения MAE моделей не уменьшилась, а выросла. Это означает, что найденные строки могут быть не мусором, а редкими или нестандартными наблюдениями, которые содержат полезную информацию для модели.

Следовательно, автоэнкодер в данной работе используется не как автоматический фильтр очистки данных, а как инструмент статистического контроля качества. Он помогает выделить наблюдения, которые требуют ручной проверки.

Нейросетевой блок проекта включает два подхода:

1. модель обучения с учителем `MLPRegressor`, которая используется для прогноза интенсивности изменения береговой бровки по подготовленным признакам;
2. автоэнкодер как инструмент обучения без учителя, который ищет нетипичные сочетания признаков без использования заранее заданного правильного ответа.

Дополнительный эксперимент показал, что механическое удаление строк, найденных автоэнкодером, не улучшает качество моделирования. Поэтому такие строки должны рассматриваться как кандидаты на предметную проверку, а не как данные для автоматического исключения.